T2 - Ciência de Dados

Base: Pima Indians Diabetes Dataset. Alvo: Outcome (0/1).

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report,
                              confusion_matrix)
import joblib
import os

if not os.path.exists('diabetes.csv'):
    !wget -q https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv

df = pd.read_csv('diabetes.csv')
df.shape

(768, 9)

In [3]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


Zeros em Glucose, BloodPressure, SkinThickness, Insulin e BMI = dados faltantes.

Vou virar NaN e imputar pela mediana.

In [4]:
cols_zero = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_zero] = df[cols_zero].replace(0, np.nan)
df.isna().sum()

,0
Pregnancies,0
Glucose,5
BloodPressure,35
SkinThickness,227
Insulin,374
BMI,11
DiabetesPedigreeFunction,0
Age,0
Outcome,0


In [5]:
X = df.drop(columns=['Outcome'])
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=42)),
])

pipeline.fit(X_train, y_train)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('model', LogisticRegression(max_iter=1000, random_state=42))])

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='accuracy')
cv_scores.mean(), cv_scores.std()

(np.float64(0.7882313741170198), np.float64(0.019056824693417025))

In [7]:
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print('Acurácia:', round(accuracy_score(y_test, y_pred), 3))
print('Precisão:', round(precision_score(y_test, y_pred), 3))
print('Recall:', round(recall_score(y_test, y_pred), 3))
print('F1:', round(f1_score(y_test, y_pred), 3))
print('AUC:', round(roc_auc_score(y_test, y_proba), 3))
print(confusion_matrix(y_test, y_pred))

Acurácia: 0.708
Precisão: 0.6
Recall: 0.5
F1: 0.545
AUC: 0.813
[[82 18]
 [27 27]]


In [8]:
joblib.dump(pipeline, 'modelo_diabetes.joblib')

['modelo_diabetes.joblib']

In [9]:
import sklearn
print(sklearn.__version__)

1.6.1


==================== TREINAR MODELO - ÁRVORE DE DECISÃO ====================


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline

pipeline_arvore = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', DecisionTreeClassifier(max_depth=500)),
])

pipeline.fit(X_train, y_train)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('model', LogisticRegression(max_iter=1000, random_state=42))])

In [15]:
import joblib

joblib.dump(pipeline_arvore, 'modelo_arvore.joblib')

['modelo_arvore.joblib']